# PART 1: LSTM (Next Category Prediction)

In [11]:
import pandas as pd

merged_df = pd.read_csv("merged_browser_ram.csv")

In [12]:
merged_df.head(10)

,timestamp,url,domain,category,browser,ram_used_mb,ram_available_mb,browser_ram_mb,time_diff,new_session,session_id,hour
0,2026-02-01 00:00:36,https://x.com/page74,x.com,social,chrome,7297.078805,8702.921195,634.550154,NaN,0,0,0
1,2026-02-01 00:02:08,https://stackoverflow.com/page13,stackoverflow.com,learning,chrome,5996.921381,10003.078619,891.922551,1.533333,0,0,0
2,2026-02-01 00:03:14,https://github.com/page3,github.com,learning,chrome,6538.321890,9461.678110,1183.595947,1.100000,0,0,0
3,2026-02-01 00:04:53,https://youtube.com/page5,youtube.com,video,chrome,7900.443030,8099.556970,632.466125,1.650000,0,0,0
4,2026-02-01 00:05:56,https://facebook.com/page68,facebook.com,social,chrome,7038.437277,8961.562723,1012.808236,1.050000,0,0,0
5,2026-02-01 00:07:50,https://medium.com/page1,medium.com,learning,chrome,5844.427020,10155.572980,437.512376,1.900000,0,0,0
6,2026-02-01 00:09:39,https://youtube.com/page82,youtube.com,video,chrome,6334.918075,9665.081925,849.381720,1.816667,0,0,0
7,2026-02-01 00:10:14,https://medium.com/page56,medium.com,learning,edge,6908.885896,9091.114104,706.713030,0.583333,0,0,0
8,2026-02-01 00:10:37,https://medium.com/page15,medium.com,learning,edge,7045.177703,8954.822297,1593.516061,0.383333,0,0,0
9,2026-02-01 01:02:37,https://stackoverflow.com/page82,stackoverflow.com,learning,edge,7044.839886,8955.160114,803.361231,52.000000,1,1,1


STEP 1: Prepare sequences

In [13]:
from sklearn.preprocessing import LabelEncoder

# Use merged_df (event-level data)
df = merged_df.copy()

# Encode categories
le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['category'])

# Sort by time
df = df.sort_values('timestamp')

# Create sequences
sequence_length = 5

X = []
y = []

categories = df['category_encoded'].values

for i in range(len(categories) - sequence_length):
    X.append(categories[i:i+sequence_length])
    y.append(categories[i+sequence_length])

import numpy as np
X = np.array(X)
y = np.array(y)

STEP 2: Train-test split

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

STEP 3: Build LSTM Model

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding

num_classes = len(le.classes_)

model = Sequential([
    Embedding(input_dim=num_classes, output_dim=8, input_length=sequence_length),
    LSTM(64),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

STEP 4: Train

In [16]:
model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.3121 - loss: 1.4917 - val_accuracy: 0.3097 - val_loss: 1.4926
Epoch 2/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.3155 - loss: 1.4873 - val_accuracy: 0.3097 - val_loss: 1.4933
Epoch 3/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.3155 - loss: 1.4872 - val_accuracy: 0.3097 - val_loss: 1.4928
Epoch 4/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.3155 - loss: 1.4869 - val_accuracy: 0.3097 - val_loss: 1.4928
Epoch 5/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.3155 - loss: 1.4870 - val_accuracy: 0.3097 - val_loss: 1.4942


STEP 5: Evaluate

In [17]:
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.3162 - loss: 1.4950
Accuracy: 0.3162158131599426


# PART 2: AUTOENCODER (Anomaly Detection)

In [18]:
session_df = pd.read_csv("session_features.csv")

In [19]:
session_df.head()

,session_id,start_time,end_time,num_events,dominant_category,avg_browser_ram,peak_browser_ram,avg_ram_used,session_duration,cluster,cluster_label
0,0,2026-02-01 00:00:36,2026-02-01 00:10:37,9,learning,882.496244,1593.516061,6767.179008,10.016667,0,Moderate sessions
1,1,2026-02-01 01:02:37,2026-02-01 01:14:21,10,social,779.629551,1158.252159,6217.307296,11.733333,1,Short casual sessions
2,2,2026-02-01 02:01:21,2026-02-01 02:04:25,4,learning,967.225601,1640.718410,6177.372339,3.066667,0,Moderate sessions
3,3,2026-02-01 02:50:25,2026-02-01 04:18:44,83,video,966.943532,1950.419004,6581.222575,88.316667,2,Heavy usage sessions
4,4,2026-02-01 04:45:44,2026-02-01 04:54:36,9,social,718.859604,1035.268334,6391.652065,8.866667,1,Short casual sessions


STEP 1: Prepare features

In [20]:
from sklearn.preprocessing import StandardScaler

features = session_df[[
    'session_duration',
    'num_events',
    'avg_browser_ram',
    'peak_browser_ram'
]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

STEP 2: Build Autoencoder

In [21]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

input_dim = X_scaled.shape[1]

input_layer = Input(shape=(input_dim,))

encoded = Dense(8, activation='relu')(input_layer)
encoded = Dense(4, activation='relu')(encoded)

decoded = Dense(8, activation='relu')(encoded)
decoded = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)

autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │            36 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 152 (608.00 B)

 Trainable params: 152 (608.00 B)

 Non-trainable params: 0 (0.00 B)

STEP 3: Train

In [22]:
autoencoder.fit(
    X_scaled, X_scaled,
    epochs=20,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.0097 - val_loss: 0.7866
Epoch 2/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7102 - val_loss: 0.5461
Epoch 3/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4339 - val_loss: 0.2928
Epoch 4/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1987 - val_loss: 0.1370
Epoch 5/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0928 - val_loss: 0.0712
Epoch 6/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0551 - val_loss: 0.0469
Epoch 7/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0412 - val_loss: 0.0365
Epoch 8/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0343 - val_loss: 0.0308
Epoch 9/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0299 - val_loss: 0.0276
Epoch 10/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0272 - val_loss: 0.0257
Epoch 11/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0253 - val_loss: 0.0240
Epoch 12/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

STEP 4: Detect anomalies

In [23]:
reconstructions = autoencoder.predict(X_scaled)

mse = np.mean(np.power(X_scaled - reconstructions, 2), axis=1)

session_df['reconstruction_error'] = mse

# Threshold (top 5% as anomalies)
threshold = np.percentile(mse, 95)

session_df['anomaly'] = session_df['reconstruction_error'] > threshold

155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 929us/step


STEP 5: View anomalies

In [24]:
anomalies = session_df[session_df['anomaly'] == True]

print(anomalies[['session_id','session_duration','peak_browser_ram']].head())

    session_id  session_duration  peak_browser_ram
22          22         26.016667       1689.423919
28          28         17.383333       1622.380906
31          31         47.916667       1679.029425
38          38         13.583333       1162.953519
52          52         20.783333       1611.671601


In [25]:
model.save("lstm_model.h5")

In [26]:

import joblib

joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [27]:
autoencoder.save("autoencoder_model.h5")
joblib.dump(scaler, "autoencoder_scaler.pkl")

['autoencoder_scaler.pkl']